# الدرس 16 - نشر وكلاء قابلين للتوسع باستخدام Microsoft Foundry

في هذا الدفتر تقوم ببناء **وكيل دعم العملاء جاهز للإنتاج** للشركة الخيالية **Contoso**. على عكس الدروس السابقة، الهدف هنا ليس حلقة تفكير الوكيل — بل كل شيء ملفوف *حولها* يجعل الوكيل آمنًا للتشغيل على نطاق واسع:

1. **استدعاء الأدوات** — عمليات بحث الطلبات وإنشاء التذاكر.
2. **RAG** — إجابات بالسياسات من قاعدة المعرفة.
3. **الذاكرة** — تذكر العميل عبر التفاعلات.
4. **توجيه النموذج** — إرسال الطلبات البسيطة إلى نموذج صغير، والطلبات المعقدة إلى نموذج كبير.
5. **تخزين الاستجابة مؤقتًا** — تقديم إجابات متكررة بدون استدعاء النموذج.
6. **الموافقة البشرية** — توقف المبالغ المستردة أعلى حد معين للموافقة.
7. **بوابة التقييم** — مجموعة اختبار غير متصلة تمنع إصدار سيء.
8. **المراقبة** — تتبع OpenTelemetry حول كل طلب.

كل قسم مستقل وقابل للتشغيل. اقرأ كل سطر — العناصر الأساسية للإنتاج محفوظة صغيرة عمدًا.


## الإعداد

قبل تشغيل هذا الدفتر، تأكد من أن لديك:

1. **مشروع Microsoft Foundry** به نموذج محادثة منشور (مثل `gpt-5-mini`).
2. **تسجيل الدخول باستخدام Azure CLI** — شغّل الأمر `az login` في الطرفية الخاصة بك.
3. **تعيين متغيرات البيئة المطلوبة:**
   - `AZURE_AI_PROJECT_ENDPOINT` — نقطة النهاية لمشروع Microsoft Foundry الخاص بك.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — اسم النموذج المنشور الخاص بك.

يستخدم قسم RAG **Azure AI Search** عند تعيين `AZURE_SEARCH_SERVICE_ENDPOINT` و `AZURE_SEARCH_API_KEY`، ويتراجع إلى البحث في الذاكرة بحيث يعمل الدفتر بدون مورد بحث.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. الأدوات

تقوم أدوات الإنتاج بأداء عمل حقيقي ضد أنظمة حقيقية. هنا نقوم بمحاكاة قاعدة بيانات للطلبات ونظام تذاكر باستخدام دوال بايثون عادية. يقوم المزخرف `@tool` بكشفها للوكيل.

لاحظ أن `issue_refund` يستخدم `approval_mode="always_require"` للمبالغ المستردة التي تتجاوز الحد — هذه هي البنية الأساسية للإنسان في الحلقة التي ننشرها لاحقًا.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — قاعدة معرفة السياسات

يجب الإجابة على أسئلة السياسة ("ما هي فترة إرجاعك؟") من مصدر موثوق، وليس من ذاكرة النموذج. نحن نغلف قاعدة معرفة صغيرة كأداة بحث.

في الإنتاج، هذه هي **Azure AI Search**؛ هنا نقدم بحثًا بالكلمات المفتاحية في الذاكرة حتى يعمل الدفتر في أي مكان، ونتحول تلقائيًا إلى Azure AI Search عندما تكون متغيرات البيئة موجودة.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. الذاكرة

وكيل الدعم الذي ينسى من يتحدث إليه هو وكيل دعم سيء. نحن نحفظ ملف تعريف صغير لكل عميل ونُدخل ملخصًا قصيرًا في تعليمات الوكيل. في بيئة الإنتاج، تكون هذه خدمة ذاكرة (انظر الدرس 13)؛ هنا، قاموس يجعل النمط مرئيًا.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## ٤ و ٥. توجيه النموذج وتخزين الاستجابة مؤقتًا

رافعتا تكلفة موصولتان إلى معالج طلب واحد:

- **التوجيه**: مصنف تكهني رخيص يقرر ما إذا كان الطلب يحتاج النموذج الصغير أم الكبير.
- **التخزين المؤقت**: تُقدّم الأسئلة المكررة المُعالجة مباشرةً من ذاكرة التخزين المؤقت بدون استدعاء النموذج.

المصنف هنا بسيط عن قصد. في الإنتاج، ستقوم بالتحقق من صحته مقابل حركة المرور وقد تستبدله بـ Foundry's Model Router.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 و 8. الوكيل، الموافقة البشرية، وقابلية المراقبة

الآن نقوم بتجميع الوكيل من الأدوات أعلاه ونغلف كل طلب في فترة OpenTelemetry. وظيفة `handle_support_request` هي معالج الطلب في الإنتاج: التخزين المؤقت → التوجيه → التتبع → التنفيذ → التخزين المؤقت.

يتم التعامل مع الموافقة البشرية بواسطة الإطار: بما أن `issue_refund` تعين `approval_mode="always_require"`، فإن التنفيذ يتوقف ويعرض طلب موافقة نقوم بحله بشكل صريح.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## ٧. بوابة التقييم

هذه هي بوابة الإصدار من الدرس: يقوم مجموعة اختبار غير متصلة بالإنترنت بتقييم الوكيل، ولا يستمر النشر إلا إذا تجاوز معدل النجاح حدًا معينًا. المُقيِّم هنا هو فحص بسيط لتداخل الكلمات المفتاحية للحفاظ على دفتر الملاحظات مكتفيًا ذاتيًا؛ في الإنتاج، ستستخدم نموذجًا لغويًا كبيرًا كقاضٍ أو مُقَيِّم إطار عمل (انظر الدرس 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## جمعها معًا: إصدار محاكاة

تعرض الخلية أدناه الحلقة الكاملة التي يصفها الدرس: تشغيل بوابة التقييم، والقيام بـ"النشر" فقط إذا اجتازت. هذا هو النمط الذي ستشغله في CI قبل ترقية نسخة الوكيل إلى خدمة Foundry Agent.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## الملخص

لقد جمعت وكيل دعم عملاء جاهز للإنتاج مع مراعاة كل الجوانب التشغيلية:

- **الأدوات، واسترجاع المعلومات (RAG)، والذاكرة** تمنح الوكيل القدرة والسياق.
- **توجيه النموذج والتخزين المؤقت** يحافظان على التحكم في التأخير والتكلفة.
- **الموافقة البشرية** تحمي الإجراءات عالية المخاطر مثل المبالغ المستردة الكبيرة.
- **بوابة التقييم** تمنع الإصدارات السيئة قبل إطلاقها.
- **التتبع** يجعل كل طلب قابلاً للرصد.

### التحدي

قم بتوسيع هذا الوكيل لـ:

1. **دعم نماذج متعددة** — أضف طبقة ثالثة "للتفكير" وقم بتوجيه التصعيدات/الشكاوى إليها.
2. **إضافة بوابات تقييم** — وسع `TEST_CASES` لتشمل سيناريوهات موافقة على الاسترداد وتأكد من أن البوابة تلتقط الانحسارات.
3. **إضافة توجيه واعي للتكلفة** — تتبع تكلفة مقدرة لكل طلب (صغير مقابل كبير مقابل التخزين المؤقت) واطبع تقرير تكلفة بعد دفعة من الاستفسارات المختلطة.

في الدرس التالي، ستأخذ الرحلة المعاكسة وتشغل وكيلًا بالكامل على جهازك الخاص باستخدام Microsoft Foundry Local و Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
